In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchmetrics
import json
from pathlib import Path
from torch.utils.data import DataLoader

In [2]:
path = 'datasets/sketches/full_simplified_ant.ndjson'

with open(path, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

In [3]:
df = pd.read_json(path, lines=True)
df.head()

,word,countrycode,timestamp,recognized,key_id,drawing
0,ant,US,2017-03-27 00:14:57.310330+00:00,True,5421013154136064,"[[[27, 17, 16, 21, 34, 50, 49, 34, 23, 17], [4..."
1,ant,US,2017-03-06 20:00:22.521560+00:00,True,4836123148812288,"[[[27, 0, 7, 40, 47, 20], [0, 41, 74, 73, 41, ..."
2,ant,US,2017-01-23 19:53:28.354530+00:00,True,5720952853757952,"[[[34, 18, 14, 4, 1, 2, 10, 18, 46, 69, 83, 89..."
3,ant,US,2017-03-14 14:52:27.521410+00:00,True,6345979559149568,"[[[59, 33, 16, 10, 61, 71, 69], [33, 36, 46, 5..."
4,ant,US,2017-01-25 21:48:31.256400+00:00,True,4704383923126272,"[[[17, 16, 19], [140, 167, 177]], [[81, 82, 87..."


In [4]:
data[0]['drawing']

[[[27, 17, 16, 21, 34, 50, 49, 34, 23, 17],
  [47, 58, 73, 81, 84, 67, 54, 46, 47, 51]],
 [[22, 0], [51, 18]],
 [[41, 46, 43], [45, 11, 0]],
 [[53, 65, 64, 69, 91, 119, 135, 148, 159, 158, 149, 126, 87, 68, 62],
  [68, 68, 58, 51, 36, 34, 38, 48, 64, 78, 85, 90, 90, 83, 73]],
 [[161, 175], [70, 69]],
 [[180, 177, 176, 187, 206, 226, 244, 250, 250, 245, 233, 207, 188, 180, 180],
  [68, 67, 61, 50, 42, 40, 48, 58, 72, 80, 87, 89, 83, 76, 71]],
 [[73, 61], [85, 113]],
 [[95, 94], [88, 126]],
 [[140, 157], [90, 118]],
 [[199, 201, 208], [90, 116, 122]],
 [[234, 242, 255], [89, 105, 112]]]

In [5]:
def TransformData(data, el_num):
    total_series = []
    for stroke in data[el_num]['drawing']:
        stroke_length = len(stroke[0])
        current_step = 0
        for pos in range(stroke_length):
            total_series += [[(stroke[0][pos] / 255.0), (stroke[1][pos] / 255.0), current_step]]
            current_step += round((1 / (stroke_length - 1)), 3) 
                
    return torch.FloatTensor(total_series)

TransformData(data, 0)[:15]

tensor([[0.1059, 0.1843, 0.0000],
        [0.0667, 0.2275, 0.1110],
        [0.0627, 0.2863, 0.2220],
        [0.0824, 0.3176, 0.3330],
        [0.1333, 0.3294, 0.4440],
        [0.1961, 0.2627, 0.5550],
        [0.1922, 0.2118, 0.6660],
        [0.1333, 0.1804, 0.7770],
        [0.0902, 0.1843, 0.8880],
        [0.0667, 0.2000, 0.9990],
        [0.0863, 0.2000, 0.0000],
        [0.0000, 0.0706, 1.0000],
        [0.1608, 0.1765, 0.0000],
        [0.1804, 0.0431, 0.5000],
        [0.1686, 0.0000, 1.0000]])

In [6]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length
        
    def __len__(self):
        return len(self.series) - self.window_length
    
    def __getitem__(self, index):
        end = index + self.window_length
        window = self.series[index:end]
        target = self.series[end]
        return window, target
    
    
def complete_dataset(data, range_num, window_length):
    total_dataset = []
    for num in range(range_num[0], range_num[1]):
        time_series = TransformData(data, num)
        time_series = TimeSeriesDataset(time_series, window_length)
        for el in range(time_series.__len__()):
            total_dataset += [time_series.__getitem__(el)]
            
    return total_dataset
    
    
    
class TimeSeriesComplete(TimeSeriesDataset):
    def __getitem__(self, index):
        if index >= len(self):
            raise IndexError('out of the range')
        return self.series[index][0], self.series[index][1] 

In [7]:
train_series = TimeSeriesComplete(complete_dataset(data, (0, 4000), 16), 16)
train_loader = DataLoader(train_series, batch_size=16, shuffle=True)

valid_series = TimeSeriesComplete(complete_dataset(data, (4001, 6000), 16), 16)
valid_loader = DataLoader(valid_series, batch_size=16)

In [8]:
class MyRnn(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.memory_cell = nn.GRU(input_size, hidden_size, num_layers=3, batch_first=True, dropout=0.05)
        self.output = nn.Linear(hidden_size, output_size)
        
    def forward(self, X):
        outputs, _last_state = self.memory_cell(X)
        return self.output(outputs[:, -1])

In [9]:
def train_model(model, optimizer, criterion, data_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            
        mean_loss = total_loss / len(data_loader)
        print(f'Epoch: {epoch}\tLoss: {mean_loss}')
        
def eval_model(model, metric, data_loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
            
        return metric.compute()

In [ ]:
# norm [-1, 1] 2m 16s, loss: 0.013977
model_1 = MyRnn(3, 32, 3).to('cuda')
optimizer = torch.optim.SGD(model_1.parameters(), lr = 0.05, momentum=0.95, nesterov=True)
huberloss = nn.HuberLoss()

train_model(model_1, optimizer, huberloss, train_loader, 10)

Epoch: 0	Loss: 0.0324032031075487
Epoch: 1	Loss: 0.01722505760174199
Epoch: 2	Loss: 0.015985868498273104
Epoch: 3	Loss: 0.015357640722748303
Epoch: 4	Loss: 0.014984904512999839
Epoch: 5	Loss: 0.014664635908207001
Epoch: 6	Loss: 0.014449538821855464
Epoch: 7	Loss: 0.014278845965081239
Epoch: 8	Loss: 0.01410931821360337
Epoch: 9	Loss: 0.0139775631316621


In [68]:
metric = torchmetrics.MeanAbsoluteError().to('cuda')
eval_model(model_1, metric, valid_loader)

tensor(0.0786, device='cuda:0')

In [ ]:
#norm [0, 1] 2m 20s loss: 0.006
model_2 = MyRnn(3, 32, 3).to('cuda')
optimizer = torch.optim.SGD(model_2.parameters(), lr = 0.05, momentum=0.95, nesterov=True)
huberloss = nn.HuberLoss()

train_model(model_2, optimizer, huberloss, train_loader, 10)

Epoch: 0	Loss: 0.02660588233733964
Epoch: 1	Loss: 0.010054440624786491
Epoch: 2	Loss: 0.007015508690411749
Epoch: 3	Loss: 0.006646477577429092
Epoch: 4	Loss: 0.006416882751486777
Epoch: 5	Loss: 0.0062602210648114226
Epoch: 6	Loss: 0.006186648966467877
Epoch: 7	Loss: 0.006093736338989313
Epoch: 8	Loss: 0.006044044371012343
Epoch: 9	Loss: 0.0060017920687138885


In [ ]:
#27% better
metric = torchmetrics.MeanAbsoluteError().to('cuda')
eval_model(model_2, metric, valid_loader)

tensor(0.0574, device='cuda:0')

In [ ]:
#norm[-1, 0] 2m 20s loss: 0.006
model_3 = MyRnn(3, 32, 3).to('cuda')
optimizer = torch.optim.SGD(model_3.parameters(), lr = 0.05, momentum=0.95, nesterov=True)
huberloss = nn.HuberLoss()

train_model(model_3, optimizer, huberloss, train_loader, 10)

Epoch: 0	Loss: 0.025029786287722677
Epoch: 1	Loss: 0.008161389514243727
Epoch: 2	Loss: 0.006988154595147381
Epoch: 3	Loss: 0.0066058639792411205
Epoch: 4	Loss: 0.0064111905611942185
Epoch: 5	Loss: 0.006268921731236848
Epoch: 6	Loss: 0.006191574905863519
Epoch: 7	Loss: 0.006085668785001145
Epoch: 8	Loss: 0.006031562834579616
Epoch: 9	Loss: 0.005965086306140173


In [ ]:
# ~same as [0,1]
metric = torchmetrics.MeanAbsoluteError().to('cuda')
eval_model(model_3, metric, valid_loader)

tensor(0.0592, device='cuda:0')

concl: model can predict the next steps to draw the ant, but now I have to build classification model between 3 classes 

In [ ]:
path = 'datasets/sketches/full_simplified_ant.ndjson'

with open(path, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]
    
train_series = TimeSeriesComplete(complete_dataset(data, (0, 4000), 16), 16)
train_loader = DataLoader(train_series, batch_size=16, shuffle=True)

valid_series = TimeSeriesComplete(complete_dataset(data, (4001, 6000), 16), 16)
valid_loader = DataLoader(valid_series, batch_size=16)

In [10]:
class TimeSeriesDataset_classification(TimeSeriesDataset):
    def __init__(self, series, window_length, class_num):
        self.series = series
        self.window_length = window_length
        self.class_num = class_num
        
    def __getitem__(self, index):
        end = index + self.window_length
        window = self.series[index:end]
        target = [0.0, 0.0, 0.0]
        target[self.class_num] = 1.0
        target = torch.FloatTensor(target)
        return window, target
    
def complete_dataset_classification(data, range_num, window_length, class_num):
    total_dataset = []
    for num in range(range_num[0], range_num[1]):
        time_series = TransformData(data, num)
        time_series = TimeSeriesDataset_classification(time_series, window_length, class_num)
        for el in range(time_series.__len__()):
            total_dataset += [time_series.__getitem__(el)]
            
    return total_dataset

In [29]:
full_data_train = data[:2000]
full_data_valid = data[2000:3000]

complete_train = complete_dataset_classification(full_data_train, (0, 2000), 16, 0)
complete_valid = complete_dataset_classification(full_data_valid, (0, 1000), 16,  0)

path = 'datasets/sketches/full_simplified_axe.ndjson'
with open(path, 'r', encoding='utf-8') as f:
    data_x = [json.loads(line) for line in f]
complete_train += complete_dataset_classification(data_x, (0, 2000), 16, 1)
complete_valid += complete_dataset_classification(data_x, (0, 1000), 16,  1)

path = 'datasets/sketches/full_simplified_bat.ndjson'
with open(path, 'r', encoding='utf-8') as f:
    data_x = [json.loads(line) for line in f]
complete_train += complete_dataset_classification(data_x, (0, 2000), 16, 2)
complete_valid += complete_dataset_classification(data_x, (0, 1000), 16,  2)

In [30]:
full_train_series = TimeSeriesComplete(complete_train, 16)
full_train_loader = DataLoader(full_train_series, batch_size=16, shuffle=True)

full_valid_series = TimeSeriesComplete(complete_valid, 16)
full_valid_loader = DataLoader(full_valid_series, batch_size=16, shuffle=True)

In [31]:
for X, y in full_valid_loader:
    print(y.shape)
    break

torch.Size([16, 3])


In [32]:
model_class = MyRnn(3, 32, 3).to('cuda')
optimizer = torch.optim.SGD(model_class.parameters(), lr = 0.05, momentum=0.95, nesterov=True)
xentropy = nn.CrossEntropyLoss()

train_model(model_class, optimizer, xentropy, full_train_loader, 10)

Epoch: 0	Loss: 0.6574222390325888
Epoch: 1	Loss: 0.6084305077370878
Epoch: 2	Loss: 0.6384619962525321
Epoch: 3	Loss: 0.7324198276562229
Epoch: 4	Loss: 0.6740519395603634
Epoch: 5	Loss: 0.752290902150643
Epoch: 6	Loss: 0.6716033477483769
Epoch: 7	Loss: 0.6500277488779422
Epoch: 8	Loss: 0.6449709110355359
Epoch: 9	Loss: 0.6366948550733816


In [34]:
metric = torchmetrics.Accuracy(task='multiclass', num_classes=3).to('cuda')

eval_model(model_class, metric, full_valid_loader)

tensor(0., device='cuda:0')

In [106]:
metric = torchmetrics.Accuracy(task='multiclass', num_classes=3).to('cuda')

model_class.eval()
with torch.no_grad():
    for X, y in full_valid_loader:
        X, y = X.to('cuda'), y.to('cuda')
        logits = torch.softmax(model_class(X), dim=1)
        y_pred = torch.nn.functional.one_hot(logits.argmax(dim=1), num_classes=logits.shape[1])
        metric.update(y_pred, y)

print(metric.compute())

tensor(0.8401, device='cuda:0')
